# Northwind — Data Dictionary
**Source:** Northwind SQLite database — enriched line-item dataset  
**Method:** `ydata_profiling` → JSON → parsed into a comprehensive, styled data dictionary  
**Rows:** each variable/column &nbsp;|&nbsp; **Columns:** all available statistical attributes per type

In [7]:
import sqlite3
import json
import pandas as pd
import numpy as np
from ydata_profiling import ProfileReport
from IPython.display import display, HTML

## 1 · Data Preparation
Same enriched line-item DataFrame as `northwind.ipynb`.

In [8]:
conn = sqlite3.connect('northwind.db')

orders_df        = pd.read_sql_query('SELECT * FROM Orders;', conn)
order_details_df = pd.read_sql_query('SELECT * FROM "Order Details";', conn)
products_df      = pd.read_sql_query('SELECT * FROM Products;', conn)
customers_df     = pd.read_sql_query('SELECT * FROM Customers;', conn)

orders_enriched_df = (
    orders_df
    .merge(order_details_df, on='OrderID',    how='left')
    .merge(products_df,      on='ProductID',  how='left')
    .merge(customers_df,     on='CustomerID', how='left')
)

df_cleaned = (
    orders_enriched_df
    .drop(columns=[c for c in orders_enriched_df.columns if c.endswith('_y')])
    .rename(columns=lambda x: x.replace('_x', ''))
)
df_cleaned['OrderRevenue'] = (
    (df_cleaned['UnitPrice'] - df_cleaned['UnitPrice'] * df_cleaned['Discount'])
    * df_cleaned['Quantity']
)
df_cleaned = df_cleaned.reset_index(names='LineItem')

print(f"Shape: {df_cleaned.shape[0]:,} rows × {df_cleaned.shape[1]} columns")
df_cleaned.dtypes.to_frame('dtype').T

Shape: 609,283 rows × 38 columns


,LineItem,OrderID,CustomerID,EmployeeID,OrderDate,RequiredDate,ShippedDate,ShipVia,Freight,ShipName,...,ContactName,ContactTitle,Address,City,Region,PostalCode,Country,Phone,Fax,OrderRevenue
dtype,int64,int64,object,int64,object,object,object,int64,float64,object,...,object,object,object,object,object,object,object,object,object,float64


## 2 · Generate Profile Report → JSON

In [9]:
profile = ProfileReport(
    df_cleaned,
    title='Northwind Data Dictionary',
    minimal=False,
    progress_bar=True,
)

profile_json_str = profile.to_json()
profile_data     = json.loads(profile_json_str)

print("Top-level keys:", list(profile_data.keys()))
print(f"Variables found: {len(profile_data.get('variables', {}))}")

Render JSON: 100%|██████████| 1/1 [00:00<00:00,  1.10it/s]

Top-level keys: ['analysis', 'time_index_analysis', 'table', 'variables', 'scatter', 'correlations', 'missing', 'alerts', 'package', 'sample', 'duplicates']
Variables found: 38


## 3 · Parse JSON → Data Dictionary

In [10]:
def _get(d, key, default=np.nan):
    """Safe dict lookup; returns default when key absent or value is None."""
    v = d.get(key)
    return default if v is None else v

def _top(vc_dict):
    """Return (top_value, top_frequency) from a value_counts dict."""
    if isinstance(vc_dict, dict) and vc_dict:
        k, v = next(iter(vc_dict.items()))
        return k, int(v)
    return np.nan, np.nan

def _pct(v, scale=100):
    """Multiply fraction by scale, round to 4 dp; return nan if nan."""
    return round(float(v) * scale, 4) if not (isinstance(v, float) and np.isnan(v)) else np.nan

def _r(v, dp=4):
    """Round float to dp decimal places; return nan if nan."""
    try:
        f = float(v)
        return round(f, dp) if not np.isnan(f) else np.nan
    except Exception:
        return np.nan


variables = profile_data.get('variables', {})
rows = []

for col_name, s in variables.items():
    vtype = _get(s, 'type', 'Unknown')

    # ── Universal ─────────────────────────────────────────────────────────────
    n         = _get(s, 'n')
    n_missing = _get(s, 'n_missing')
    p_missing = _get(s, 'p_missing')
    n_distinct = _get(s, 'n_distinct')
    p_distinct = _get(s, 'p_distinct')
    n_unique  = _get(s, 'n_unique')
    p_unique  = _get(s, 'p_unique')
    is_unique = _get(s, 'is_unique')
    mem_bytes = _get(s, 'memory_size')

    n_count = (int(n) - int(n_missing)) if not (isinstance(n, float) and np.isnan(n)) and \
               not (isinstance(n_missing, float) and np.isnan(n_missing)) else np.nan

    # top value (works for both Numeric & Text)
    top_val, top_freq = _top(_get(s, 'value_counts_without_nan', {}))

    # ── Numeric-specific ──────────────────────────────────────────────────────
    mean     = _r(_get(s, 'mean'))
    std      = _r(_get(s, 'std'))
    variance = _r(_get(s, 'variance'))
    cv       = _r(_get(s, 'cv'))
    skewness = _r(_get(s, 'skewness'))
    kurtosis = _r(_get(s, 'kurtosis'))
    vmin     = _get(s, 'min')
    vmax     = _get(s, 'max')
    vrange   = _r(_get(s, 'range'))
    vsum     = _r(_get(s, 'sum'), dp=2)
    mad      = _r(_get(s, 'mad'))
    iqr      = _r(_get(s, 'iqr'))
    p05      = _r(_get(s, '5%'))
    p25      = _r(_get(s, '25%'))
    p50      = _r(_get(s, '50%'))
    p75      = _r(_get(s, '75%'))
    p95      = _r(_get(s, '95%'))
    n_zeros  = _get(s, 'n_zeros')
    p_zeros  = _get(s, 'p_zeros')
    n_neg    = _get(s, 'n_negative')
    p_neg    = _get(s, 'p_negative')
    n_inf    = _get(s, 'n_infinite')
    mono     = _get(s, 'monotonic')

    # ── Text-specific ─────────────────────────────────────────────────────────
    min_len    = _get(s, 'min_length')
    max_len    = _get(s, 'max_length')
    mean_len   = _r(_get(s, 'mean_length'))
    median_len = _r(_get(s, 'median_length'))
    n_chars    = _get(s, 'n_characters')
    n_chars_d  = _get(s, 'n_characters_distinct')

    pd_dtype = str(df_cleaned[col_name].dtype) if col_name in df_cleaned.columns else '—'

    rows.append({
        # ── identity ──
        'Variable':          col_name,
        'Type':              vtype,
        'Pandas dtype':      pd_dtype,
        'Memory (bytes)':    int(mem_bytes) if not isinstance(mem_bytes, float) else np.nan,
        # ── completeness ──
        'Total Rows':        int(n) if not isinstance(n, float) else np.nan,
        'Count (non-null)':  int(n_count) if not isinstance(n_count, float) else n_count,
        '# Missing':         int(n_missing) if not isinstance(n_missing, float) else np.nan,
        '% Missing':         _pct(p_missing),
        # ── uniqueness ──
        '# Distinct':        int(n_distinct) if not isinstance(n_distinct, float) else np.nan,
        '% Distinct':        _pct(p_distinct),
        '# Unique (exact)':  int(n_unique) if not isinstance(n_unique, float) else np.nan,
        '% Unique (exact)':  _pct(p_unique),
        'Is Unique':         bool(is_unique) if not isinstance(is_unique, float) else np.nan,
        # ── central tendency (Numeric) ──
        'Mean':              mean,
        'Median (50%)':      p50,
        # ── spread (Numeric) ──
        'Std Dev':           std,
        'Variance':          variance,
        'MAD':               mad,
        'CV':                cv,
        'IQR':               iqr,
        # ── shape (Numeric) ──
        'Skewness':          skewness,
        'Kurtosis':          kurtosis,
        # ── range / quantiles (Numeric) ──
        'Min':               vmin,
        '5th Pct':           p05,
        '25th Pct':          p25,
        '75th Pct':          p75,
        '95th Pct':          p95,
        'Max':               vmax,
        'Range':             vrange,
        # ── aggregate (Numeric) ──
        'Sum':               vsum,
        # ── special counts (Numeric) ──
        '# Zeros':           int(n_zeros) if not isinstance(n_zeros, float) else np.nan,
        '% Zeros':           _pct(p_zeros),
        '# Negative':        int(n_neg) if not isinstance(n_neg, float) else np.nan,
        '% Negative':        _pct(p_neg),
        '# Infinite':        int(n_inf) if not isinstance(n_inf, float) else np.nan,
        'Monotonic':         mono,
        # ── text-specific ──
        'Min Length':        int(min_len) if not isinstance(min_len, float) else np.nan,
        'Max Length':        int(max_len) if not isinstance(max_len, float) else np.nan,
        'Mean Length':       mean_len,
        'Median Length':     median_len,
        '# Characters':      int(n_chars) if not isinstance(n_chars, float) else np.nan,
        '# Distinct Chars':  int(n_chars_d) if not isinstance(n_chars_d, float) else np.nan,
        # ── categorical summary ──
        'Top Value':         top_val,
        'Top Freq':          int(top_freq) if not (isinstance(top_freq, float) and np.isnan(top_freq)) else np.nan,
    })

dict_df = pd.DataFrame(rows).set_index('Variable')
print(f"Data dictionary: {dict_df.shape[0]} variables × {dict_df.shape[1]} attributes")
dict_df.head(3)

Data dictionary: 38 variables × 43 attributes


,Type,Pandas dtype,Memory (bytes),Total Rows,Count (non-null),# Missing,% Missing,# Distinct,% Distinct,# Unique (exact),...,# Infinite,Monotonic,Min Length,Max Length,Mean Length,Median Length,# Characters,# Distinct Chars,Top Value,Top Freq
Variable,,,,,,,,,,,,,,,,,,,,,
LineItem,Numeric,int64,4874396,609283,609283,0,0.0,609283,100.0000,609283,...,0.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,0,1
OrderID,Numeric,int64,4874396,609283,609283,0,0.0,16282,2.6723,334,...,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,15878,77
CustomerID,Text,object,4874396,609283,609283,0,0.0,93,0.0153,0,...,NaN,NaN,5.0,5.0,5.0,5.0,3046415.0,28.0,BSBEV,8287


## 4 · Styled Data Dictionary Table

In [11]:
# ── Colour palette ────────────────────────────────────────────────────────────
BG_HEADER   = '#0f172a'   # deep navy
BG_ROW_IDX  = '#1e293b'   # dark slate (row index)
BG_ODD      = '#f8fafc'
BG_EVEN     = '#ffffff'
BORDER      = '#e2e8f0'
FONT        = '"Inter", "Segoe UI", Arial, sans-serif'

BADGE = {
    'Numeric':     '#1d4ed8',  # blue
    'Text':        '#0f766e',  # teal
    'DateTime':    '#7c3aed',  # violet
    'Boolean':     '#b45309',  # amber
    'Categorical': '#15803d',  # green
    'Unsupported': '#64748b',  # slate
}

# ── Column groups (order matters) ────────────────────────────────────────────
GROUPS = [
    ('Identity',            ['Type', 'Pandas dtype', 'Memory (bytes)']),
    ('Completeness',        ['Total Rows', 'Count (non-null)', '# Missing', '% Missing']),
    ('Uniqueness',          ['# Distinct', '% Distinct', '# Unique (exact)', '% Unique (exact)', 'Is Unique']),
    ('Central Tendency',    ['Mean', 'Median (50%)']),
    ('Spread',              ['Std Dev', 'Variance', 'MAD', 'CV', 'IQR']),
    ('Shape',               ['Skewness', 'Kurtosis']),
    ('Range / Quantiles',   ['Min', '5th Pct', '25th Pct', '75th Pct', '95th Pct', 'Max', 'Range']),
    ('Aggregate',           ['Sum']),
    ('Special Counts',      ['# Zeros', '% Zeros', '# Negative', '% Negative', '# Infinite', 'Monotonic']),
    ('Text Metrics',        ['Min Length', 'Max Length', 'Mean Length', 'Median Length',
                              '# Characters', '# Distinct Chars']),
    ('Most Frequent',       ['Top Value', 'Top Freq']),
]


# ── Styling helpers ───────────────────────────────────────────────────────────
def badge_style(val):
    bg = BADGE.get(str(val), BADGE['Unsupported'])
    return (f'background:{bg};color:#fff;border-radius:4px;'
            f'padding:2px 7px;font-size:0.75em;font-weight:700;'
            f'letter-spacing:0.03em')


def heat_missing(val):
    """White → red gradient on % Missing."""
    if pd.isna(val):
        return ''
    t = min(float(val) / 100, 1.0)
    g = int(255 * (1 - t * 0.85))
    b = int(255 * (1 - t * 0.85))
    fg = '#fff' if t > 0.55 else '#1a1a1a'
    return f'background-color:rgb(255,{g},{b});color:{fg}'


def col_gradient(series, lo='#eff6ff', hi='#1d4ed8'):
    """Per-column blue gradient for numeric spread columns."""
    nums = pd.to_numeric(series, errors='coerce')
    mn, mx = nums.min(), nums.max()
    styles = []
    for v in nums:
        if pd.isna(v) or mx == mn:
            styles.append('')
        else:
            t = (v - mn) / (mx - mn)
            # interpolate between lo (#eff6ff) and hi (#1d4ed8)
            r = int(0xef + (0x1d - 0xef) * t)
            g = int(0xf6 + (0x4e - 0xf6) * t)
            b = int(0xff + (0xd8 - 0xff) * t)
            fg = '#fff' if t > 0.55 else '#1a1a1a'
            styles.append(f'background-color:rgb({r},{g},{b});color:{fg}')
    return styles


# ── Number format spec ────────────────────────────────────────────────────────
FMT = {
    '% Missing':     '{:.2f}%',
    '% Distinct':    '{:.2f}%',
    '% Unique (exact)': '{:.2f}%',
    '% Zeros':       '{:.2f}%',
    '% Negative':    '{:.2f}%',
    'Mean':          '{:,.4f}',
    'Median (50%)':  '{:,.4f}',
    'Std Dev':       '{:,.4f}',
    'Variance':      '{:,.4f}',
    'MAD':           '{:,.4f}',
    'CV':            '{:,.4f}',
    'IQR':           '{:,.4f}',
    'Skewness':      '{:,.4f}',
    'Kurtosis':      '{:,.4f}',
    '5th Pct':       '{:,.4f}',
    '25th Pct':      '{:,.4f}',
    '75th Pct':      '{:,.4f}',
    '95th Pct':      '{:,.4f}',
    'Range':         '{:,.4f}',
    'Sum':           '{:,.2f}',
    'Mean Length':   '{:.2f}',
    'Median Length': '{:.2f}',
    'Total Rows':    '{:,}',
    'Count (non-null)': '{:,}',
    '# Missing':     '{:,}',
    '# Distinct':    '{:,}',
    '# Unique (exact)': '{:,}',
    '# Zeros':       '{:,}',
    '# Negative':    '{:,}',
    '# Infinite':    '{:,}',
    'Memory (bytes)':'{:,}',
    '# Characters':  '{:,}',
    '# Distinct Chars': '{:,}',
    'Top Freq':      '{:,}',
}

gradient_cols = [
    c for c in ['Mean', 'Median (50%)', 'Std Dev', 'Variance', 'MAD',
                 'CV', 'IQR', 'Skewness', 'Kurtosis', 'Sum', 'Range']
    if c in dict_df.columns
]

# ── Build Styler ──────────────────────────────────────────────────────────────
styler = (
    dict_df.style
    .map(badge_style,   subset=['Type'])
    .map(heat_missing,  subset=['% Missing'])
    .apply(col_gradient, subset=gradient_cols)
    .format(FMT, na_rep='—')
    .set_table_styles([
        {'selector': 'table',
         'props': [('border-collapse', 'collapse'),
                   ('font-family',     FONT),
                   ('font-size',       '12.5px'),
                   ('width',           '100%'),
                   ('border-spacing',  '0')]},
        # column headers
        {'selector': 'thead th',
         'props': [('background-color', BG_HEADER),
                   ('color',            '#cbd5e1'),
                   ('padding',          '9px 11px'),
                   ('text-align',       'center'),
                   ('font-weight',      '600'),
                   ('font-size',        '11.5px'),
                   ('letter-spacing',   '0.03em'),
                   ('border',           f'1px solid #1e3a5f'),
                   ('white-space',      'nowrap'),
                   ('position',         'sticky'),
                   ('top',              '32px'),
                   ('z-index',          '2')]},
        # index (Variable names)
        {'selector': 'tbody th',
         'props': [('background-color', BG_ROW_IDX),
                   ('color',            '#e2e8f0'),
                   ('font-weight',      '700'),
                   ('font-size',        '12px'),
                   ('padding',          '8px 14px'),
                   ('border',           f'1px solid #334155'),
                   ('white-space',      'nowrap'),
                   ('min-width',        '145px'),
                   ('position',         'sticky'),
                   ('left',             '0'),
                   ('z-index',          '1')]},
        # data cells
        {'selector': 'td',
         'props': [('padding',      '7px 11px'),
                   ('text-align',   'center'),
                   ('border',       f'1px solid {BORDER}'),
                   ('white-space',  'nowrap'),
                   ('vertical-align', 'middle')]},
        # alternating rows
        {'selector': 'tbody tr:nth-child(odd)  td', 'props': [('background-color', BG_ODD)]},
        {'selector': 'tbody tr:nth-child(even) td', 'props': [('background-color', BG_EVEN)]},
        # hover
        {'selector': 'tbody tr:hover td',  'props': [('filter', 'brightness(0.94)')]},
        {'selector': 'tbody tr:hover th',  'props': [('filter', 'brightness(1.15)')]},
        # caption
        {'selector': 'caption',
         'props': [('caption-side', 'top'),
                   ('text-align',   'left'),
                   ('padding',      '0 0 8px 0')]},
    ])
    .set_caption(
        '<div style="font-family:{f};font-size:18px;font-weight:800;'
        'color:#0f172a;padding:14px 0 2px;">'
        'Northwind Enriched Dataset — Data Dictionary'
        '</div>'
        '<div style="font-size:11.5px;color:#64748b;padding-bottom:10px;">'
        'Each row = one variable. '
        'Numeric columns show distribution statistics; '
        'Text columns show length & character metrics; '
        'both show completeness and top-frequency summaries. '
        'Missing cells (—) indicate the attribute is N/A for that type.'
        '</div>'.format(f=FONT)
    )
)


# ── Inject column-group header row above the column names ─────────────────────
def _group_header_row(groups, df):
    """Build the group-label <tr> that spans multiple <th> columns."""
    # leading cell for the index column
    cells = [
        '<th rowspan="2" style="'
        'background:#0f172a;color:#94a3b8;font-size:10px;'
        'letter-spacing:0.08em;text-transform:uppercase;'
        'padding:6px 14px;border:1px solid #1e3a5f;'
        'font-weight:700;text-align:center;'
        'position:sticky;left:0;z-index:3;'
        '">VARIABLE</th>'
    ]
    GROUP_COLORS = [
        '#1e3a5f','#164e3f','#3b1f63','#4a1c1c',
        '#1c3a4a','#2a2a10','#1a3a2a','#1f2a40',
        '#2f1f40','#1a3030','#10302a',
    ]
    ci = 0
    for grp, cols in groups:
        present = [c for c in cols if c in df.columns]
        if not present:
            continue
        bg = GROUP_COLORS[ci % len(GROUP_COLORS)]
        cells.append(
            f'<th colspan="{len(present)}" style="'
            f'background:{bg};color:#93c5fd;'
            f'font-size:10px;letter-spacing:0.06em;text-transform:uppercase;'
            f'padding:5px 10px;border:1px solid #1e3a5f;'
            f'font-weight:800;text-align:center;">'
            f'{grp}</th>'
        )
        ci += 1
    return '<tr>' + ''.join(cells) + '</tr>'


raw_html   = styler.to_html()
group_row  = _group_header_row(GROUPS, dict_df)

# Insert group row as first row inside <thead>
thead_pos  = raw_html.find('<thead>')
insert_at  = thead_pos + len('<thead>')
final_html = raw_html[:insert_at] + group_row + raw_html[insert_at:]

# Wrap in scrollable container
final_html = (
    '<div id="data-dict-wrapper" style="'
    'overflow-x:auto;overflow-y:auto;max-height:750px;'
    'border:1px solid #e2e8f0;border-radius:10px;'
    'box-shadow:0 4px 20px rgba(0,0,0,0.10);'
    'background:#fff;margin:8px 0;">'
    + final_html +
    '</div>'
)

display(HTML(final_html))

## 5 · Export to Excel (styled)

In [12]:
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

OUT_PATH = 'northwind_data_dictionary.xlsx'

# ── Palette (hex without #) ───────────────────────────────────────────────────
C = dict(
    hdr_bg='0f172a', hdr_fg='cbd5e1', grp_fg='93c5fd',
    idx_bg='1e293b', idx_fg='e2e8f0',
    odd_bg='f8fafc', even_bg='ffffff',
    num_bg='1d4ed8', txt_bg='0f766e', dt_bg='7c3aed',
    bool_bg='b45309', cat_bg='15803d', unk_bg='64748b',
)
GROUP_BG  = ['1e3a5f','164e3f','3b1f63','4a1c1c','1c3a4a',
             '2a2a10','1a3a2a','1f2a40','2f1f40','1a3030','10302a']
TYPE_BADGE = {'Numeric': C['num_bg'], 'Text': C['txt_bg'],
              'DateTime': C['dt_bg'], 'Boolean': C['bool_bg'],
              'Categorical': C['cat_bg']}

# ── Helpers ───────────────────────────────────────────────────────────────────
def fill(h):       return PatternFill('solid', fgColor=h)
def font(h='1a1a2e', bold=False, sz=10):
    return Font(name='Calibri', color=h, bold=bold, size=sz)
def align(h='center', v='center', wrap=False):
    return Alignment(horizontal=h, vertical=v, wrap_text=wrap)
def border():
    s = Side(style='thin', color='d0d7de')
    return Border(left=s, right=s, top=s, bottom=s)
def lerp_color(t, lo=(239,246,255), hi=(29,78,216)):
    r=int(lo[0]+(hi[0]-lo[0])*t); g=int(lo[1]+(hi[1]-lo[1])*t); b=int(lo[2]+(hi[2]-lo[2])*t)
    return f'{r:02X}{g:02X}{b:02X}'
def missing_color(pct):
    t = min(float(pct)/100, 1.0)*0.85
    g = int(255*(1-t)); b = int(255*(1-t))
    return f'FF{g:02X}{b:02X}'

# ── Workbook setup ────────────────────────────────────────────────────────────
wb = openpyxl.Workbook()
ws = wb.active
ws.title = 'Data Dictionary'
ws.sheet_properties.tabColor = '1d4ed8'

export_df = dict_df.copy()
all_cols  = list(export_df.columns)

# ── Row 1: Group headers (merged cells) ───────────────────────────────────────
ws.merge_cells(start_row=1, start_column=1, end_row=2, end_column=1)
vc = ws.cell(row=1, column=1, value='VARIABLE')
vc.fill = fill(C['hdr_bg']); vc.font = font(C['grp_fg'], bold=True, sz=9)
vc.alignment = align('center','center'); vc.border = border()

col_offset = 2
for gi, (grp_name, grp_cols) in enumerate(GROUPS):
    present = [c for c in grp_cols if c in export_df.columns]
    if not present: continue
    s, e = col_offset, col_offset + len(present) - 1
    if s < e:
        ws.merge_cells(start_row=1, start_column=s, end_row=1, end_column=e)
    # Only set value/style on the top-left cell (merged cells are read-only)
    gc = ws.cell(row=1, column=s, value=grp_name.upper())
    gc.fill = fill(GROUP_BG[gi % len(GROUP_BG)])
    gc.font = font(C['grp_fg'], bold=True, sz=9)
    gc.alignment = align('center','center')
    gc.border = border()
    col_offset = e + 1

# ── Row 2: Column headers ─────────────────────────────────────────────────────
for ci, col_name in enumerate(all_cols):
    cell = ws.cell(row=2, column=ci+2, value=col_name)
    cell.fill = fill(C['hdr_bg']); cell.font = font(C['hdr_fg'], bold=True, sz=9)
    cell.alignment = align('center','center', wrap=True); cell.border = border()

# ── Pre-compute gradient ranges ───────────────────────────────────────────────
GRADIENT_COLS = ['Mean','Median (50%)','Std Dev','Variance','MAD',
                 'CV','IQR','Skewness','Kurtosis','Sum','Range']
col_ranges = {}
for c in GRADIENT_COLS:
    if c in export_df.columns:
        nums = pd.to_numeric(export_df[c], errors='coerce').dropna()
        if not nums.empty and nums.max() != nums.min():
            col_ranges[c] = (nums.min(), nums.max())

missing_ci = (all_cols.index('% Missing')+2) if '% Missing' in all_cols else None
type_ci    = (all_cols.index('Type')+2)       if 'Type'      in all_cols else None

NUM_FMT = {
    '% Missing':'0.00"%"', '% Distinct':'0.00"%"', '% Unique (exact)':'0.00"%"',
    '% Zeros':'0.00"%"',   '% Negative':'0.00"%"',
    'Mean':'#,##0.0000',   'Median (50%)':'#,##0.0000', 'Std Dev':'#,##0.0000',
    'Variance':'#,##0.0000','MAD':'#,##0.0000','CV':'#,##0.0000',
    'IQR':'#,##0.0000',    'Skewness':'#,##0.0000',     'Kurtosis':'#,##0.0000',
    '5th Pct':'#,##0.0000','25th Pct':'#,##0.0000',
    '75th Pct':'#,##0.0000','95th Pct':'#,##0.0000',
    'Range':'#,##0.0000',  'Sum':'#,##0.00',
    'Mean Length':'0.00',  'Median Length':'0.00',
    'Total Rows':'#,##0',  'Count (non-null)':'#,##0',  '# Missing':'#,##0',
    '# Distinct':'#,##0',  '# Unique (exact)':'#,##0',  '# Zeros':'#,##0',
    '# Negative':'#,##0',  '# Infinite':'#,##0',        'Memory (bytes)':'#,##0',
    '# Characters':'#,##0','# Distinct Chars':'#,##0',   'Top Freq':'#,##0',
}

# ── Data rows ─────────────────────────────────────────────────────────────────
for ri, (var_name, row_data) in enumerate(export_df.iterrows()):
    er     = ri + 3
    row_bg = C['odd_bg'] if ri % 2 == 0 else C['even_bg']

    idx = ws.cell(row=er, column=1, value=var_name)
    idx.fill = fill(C['idx_bg']); idx.font = font(C['idx_fg'], bold=True, sz=10)
    idx.alignment = align('left','center'); idx.border = border()

    for ci, col_name in enumerate(all_cols):
        ec  = ci + 2
        raw = row_data[col_name]
        val = None if (pd.isna(raw) if not isinstance(raw, (list, dict)) else False) \
              else (raw.item() if hasattr(raw, 'item') else raw)

        cell = ws.cell(row=er, column=ec, value=val)
        cell.alignment = align('center','center'); cell.border = border()
        cell.font = font('1a1a2e', sz=10)
        if col_name in NUM_FMT and val is not None:
            cell.number_format = NUM_FMT[col_name]

        if ec == type_ci and isinstance(val, str):
            cell.fill = fill(TYPE_BADGE.get(val, C['unk_bg']))
            cell.font = font('ffffff', bold=True, sz=10)
        elif ec == missing_ci and val is not None:
            cell.fill = fill(missing_color(val))
            cell.font = font('ffffff' if val > 55 else '1a1a1a', sz=10)
        elif col_name in col_ranges and val is not None:
            mn, mx = col_ranges[col_name]
            t = (float(val) - mn) / (mx - mn)
            cell.fill = fill(lerp_color(t))
            cell.font = font('ffffff' if t > 0.55 else '1a1a2e', sz=10)
        else:
            cell.fill = fill(row_bg)

# ── Freeze panes, row heights, column widths ──────────────────────────────────
ws.freeze_panes = 'B3'
ws.row_dimensions[1].height = 20
ws.row_dimensions[2].height = 34
for r in range(3, 3 + len(export_df)):
    ws.row_dimensions[r].height = 17

ws.column_dimensions['A'].width = 22
for ci, col_name in enumerate(all_cols):
    letter  = get_column_letter(ci + 2)
    lengths = [len(str(col_name))] + [len(str(v)) for v in export_df[col_name].dropna()]
    ws.column_dimensions[letter].width = min(max(max(lengths) * 1.05 + 2, 9), 26)

wb.save(OUT_PATH)
print(f"Saved → {OUT_PATH}")
print(f"  {len(export_df)} variables × {len(all_cols)} attributes")
print(f"  Freeze panes at B3  |  Sheet tab: blue")

Saved → northwind_data_dictionary.xlsx
  38 variables × 43 attributes
  Freeze panes at B3  |  Sheet tab: blue
